# ДЗ 7. Часові ряди та напівструктуровані дані в ML

Автор: Костромицька Ірина Вікторівна

У роботі я роблю дві речі. Спочатку з даних про поїздки таксі Нью-Йорка рахую ознаки для ML (lag, rolling і expanding) через віконні функції. Потім будую таблицю подій з JSONB, роблю запити по ній і дивлюсь через `EXPLAIN`, як працюють індекси.

Дані для часової частини: **NYC TLC Yellow Taxi Trip Records**, джерело https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page. Беру один місяць, березень 2024. Події для JSONB я генерую сама.

## Завдання 1. Setup і завантаження даних

Ставлю бібліотеки, запускаю PostgreSQL через `pgserver` і перевіряю підключення.

In [1]:
%pip install -q psycopg2-binary sqlalchemy pandas pyarrow

In [2]:
import subprocess
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

PGDATA = Path("/tmp/hw7_pg")


def run_command(command: str) -> str:
    """Запускає команду в bash і повертає її вивід."""
    result = subprocess.run(command, shell=True, executable="/bin/bash", capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Команда не виконалась: {command}\n{result.stderr}")
    return result.stdout


def start_pgserver() -> str:
    """Запускає PostgreSQL через pgserver і повертає рядок підключення."""
    pip_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pgserver"], capture_output=True, text=True
    )
    if pip_result.returncode == 0:
        import pgserver

        return pgserver.get_server(PGDATA, cleanup_mode="stop").get_uri()

    # pgserver є лише для Python 3.9-3.12, тому в новому Colab беру окремий Python 3.12
    venv = Path("/tmp/hw7_py312")
    run_command(f"{sys.executable} -m pip install -q uv")
    run_command(f"{sys.executable} -m uv venv --python 3.12 {venv}")
    run_command(f"{sys.executable} -m uv pip install -q --python {venv}/bin/python pgserver")
    start_script = "import sys, pgserver; print(pgserver.get_server(sys.argv[1], cleanup_mode=None).get_uri())"
    output = run_command(f'{venv}/bin/python -c "{start_script}" {PGDATA}')
    return output.strip().splitlines()[-1]


pg_uri = start_pgserver()
# SQLAlchemy 2 сам би взяв драйвер psycopg 3, а ми ставимо psycopg2, тому вказую його явно
engine = create_engine(pg_uri.replace("postgresql://", "postgresql+psycopg2://", 1), future=True)

# одне підключення для всіх запитів, щоб усе виконувалось в одній сесії
conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")
print(conn.execute(text("SELECT version()")).scalar_one())


def run_sql(query: str):
    """Виконує SQL. Якщо запит повертає рядки, показує їх таблицею."""
    result = conn.execute(text(query))
    if result.returns_rows:
        return pd.DataFrame(result.fetchall(), columns=list(result.keys()))


def explain(query: str) -> None:
    """Виконує EXPLAIN і друкує план повністю, щоб він лишився у виводі."""
    rows = conn.execute(text(query)).fetchall()
    print("\n".join(row[0] for row in rows))

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


### Варіант A. Реальні дані NYC Taxi

Качаю parquet-файл з поїздками за березень 2024, беру тільки потрібні колонки і прибираю явно погані рядки (від'ємна ціна, нульова або занадто велика відстань). Файл великий, тому лишаю випадкові 500 000 поїздок. Випадкові, а не перші, бо тоді в даних будуть усі дні місяця. Потім кладу їх у таблицю `nyc_taxi_raw`.

Якщо файл не завантажиться, нижче є запасний варіант B з синтетичними даними.

In [3]:
from urllib.request import urlretrieve

URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet"
parquet_path = "yellow_tripdata_2024-03.parquet"

columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "passenger_count",
]

taxi_source = None  # заповниться, якщо реальні дані завантажились

try:
    urlretrieve(URL, parquet_path)
except OSError as error:
    print("Не вдалось завантажити файл, перейду на stub-схему нижче:", error)
else:
    df = pd.read_parquet(parquet_path, columns=columns)
    print("Рядків у файлі:", len(df))

    # у файлі за березень трапляються поїздки з інших дат, їх прибираю
    df = df[(df["tpep_pickup_datetime"] >= "2024-03-01") & (df["tpep_pickup_datetime"] < "2024-04-01")]

    # базове очищення
    df = df[(df["fare_amount"] > 0) & (df["fare_amount"] < 500)]
    df = df[(df["trip_distance"] > 0) & (df["trip_distance"] < 100)]
    df = df[df["PULocationID"].notna()]

    # беру випадкові 500 000 рядків, а не перші, щоб у вибірці були всі дні місяця
    df = df.sample(500_000, random_state=42)

    df.to_sql("nyc_taxi_raw", engine, if_exists="replace", index=False, chunksize=10_000)
    taxi_source = "NYC TLC Yellow Taxi, березень 2024 (вибірка 500 000 поїздок)"

print("Рядків після очищення і вибірки:", len(df) if taxi_source else 0)

Рядків у файлі: 3582628
Рядків після очищення і вибірки: 500000


In [4]:
if taxi_source:
    display(run_sql("SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw"))

,n_rows
0,500000


Тепер роблю денну агрегацію `t7_hw_taxi_daily`: для кожної зони посадки і дня рахую кількість поїздок, середню ціну і сумарну відстань. День беру за локальним часом Нью-Йорка, тому `ts_day` має тип `DATE`. Пару `pickup_zone_id` і `ts_day` роблю первинним ключем.

In [5]:
if taxi_source:
    run_sql("""
    DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;

    CREATE TABLE t7_hw_taxi_daily AS
    SELECT
        "PULocationID"::INTEGER                                   AS pickup_zone_id,
        DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
        COUNT(*)::INTEGER                                         AS trip_count,
        AVG(fare_amount)::NUMERIC(10, 2)                          AS avg_fare,
        SUM(trip_distance)::NUMERIC(12, 2)                        AS total_distance
    FROM nyc_taxi_raw
    WHERE fare_amount > 0
      AND fare_amount < 500
      AND trip_distance > 0
      AND trip_distance < 100
      AND "PULocationID" IS NOT NULL
    GROUP BY
        "PULocationID"::INTEGER,
        DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

    ALTER TABLE t7_hw_taxi_daily
        ADD PRIMARY KEY (pickup_zone_id, ts_day);
    """)

Окрема таблиця з типом `TIMESTAMPTZ` показує, як зберігали б час у реальній системі. У файлі час записаний як локальний час Нью-Йорка без часової зони, тому я додаю зону `America/New_York`.

In [6]:
if taxi_source:
    run_sql("""
    DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;

    CREATE TABLE t7_hw_taxi_trips_typed AS
    SELECT
        "PULocationID"::INTEGER AS pickup_zone_id,
        "DOLocationID"::INTEGER AS dropoff_zone_id,
        (tpep_pickup_datetime::TIMESTAMP AT TIME ZONE 'America/New_York')  AS pickup_at,
        (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS dropoff_at,
        trip_distance::NUMERIC(10, 2) AS trip_distance,
        fare_amount::NUMERIC(10, 2)   AS fare_amount,
        passenger_count::INTEGER      AS passenger_count
    FROM nyc_taxi_raw
    WHERE fare_amount > 0
      AND fare_amount < 500
      AND trip_distance > 0
      AND trip_distance < 100;
    """)

In [7]:
if taxi_source:
    display(run_sql("""
SELECT pickup_at, pg_typeof(pickup_at) AS pickup_type
FROM t7_hw_taxi_trips_typed
LIMIT 5
    """))

,pickup_at,pickup_type
0,2024-03-16 16:52:24+00:00,timestamp with time zone
1,2024-03-10 05:12:10+00:00,timestamp with time zone
2,2024-03-22 15:40:48+00:00,timestamp with time zone
3,2024-03-02 23:05:51+00:00,timestamp with time zone
4,2024-03-27 00:22:26+00:00,timestamp with time zone


### Варіант B. Stub-схема

Це запасний варіант, на випадок якщо реальні дані не завантажились. Він створює 60 днів по 5 зон, тобто 300 рядків. Дата фіксована (`2024-03-31`), а не `CURRENT_DATE`, тому результат однаковий при кожному запуску. Якщо варіант A спрацював, ця клітинка нічого не робить.

In [8]:
if taxi_source is None:
    run_sql("""
    DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;

    CREATE TABLE t7_hw_taxi_daily (
        pickup_zone_id  INTEGER       NOT NULL,
        ts_day          DATE          NOT NULL,
        trip_count      INTEGER       NOT NULL CHECK (trip_count >= 0),
        avg_fare        NUMERIC(10, 2) NOT NULL CHECK (avg_fare >= 0),
        total_distance  NUMERIC(12, 2) NOT NULL CHECK (total_distance >= 0),
        PRIMARY KEY (pickup_zone_id, ts_day)
    );

    INSERT INTO t7_hw_taxi_daily (pickup_zone_id, ts_day, trip_count, avg_fare, total_distance)
    SELECT
        z.zone_id,
        DATE '2024-03-31' - (59 - n)                         AS ts_day,
        500 + (n * 7) + z.zone_id * 50 + MOD(n, 5) * 30      AS trip_count,
        (12.5 + z.zone_id * 0.7 + MOD(n, 4) * 0.4)::NUMERIC(10, 2) AS avg_fare,
        (1500 + n * 25 + z.zone_id * 120)::NUMERIC(12, 2)    AS total_distance
    FROM generate_series(0, 59) AS n
    CROSS JOIN (VALUES (1), (2), (3), (4), (5)) AS z(zone_id);
    """)
    taxi_source = "stub-схема (60 днів, 5 зон, синтетичні дані)"
else:
    print("Реальні дані вже завантажені, stub не потрібен")

Реальні дані вже завантажені, stub не потрібен


Перевіряю, що `t7_hw_taxi_daily` створена і в ній понад 100 рядків.

In [9]:
print("Джерело даних:", taxi_source)
daily_info = run_sql("""
SELECT
    COUNT(*)                       AS n_rows,
    COUNT(DISTINCT pickup_zone_id) AS n_zones,
    MIN(ts_day)                    AS first_day,
    MAX(ts_day)                    AS last_day
FROM t7_hw_taxi_daily
""")
assert daily_info.loc[0, "n_rows"] > 100, "У t7_hw_taxi_daily має бути понад 100 рядків"
daily_info

Джерело даних: NYC TLC Yellow Taxi, березень 2024 (вибірка 500 000 поїздок)


,n_rows,n_zones,first_day,last_day
0,5445,250,2024-03-01,2024-03-31


In [10]:
run_sql("""
SELECT *
FROM t7_hw_taxi_daily
ORDER BY pickup_zone_id, ts_day
LIMIT 10
""")

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance
0,1,2024-03-13,1,117.00,0.09
1,1,2024-03-14,1,29.60,2.93
2,1,2024-03-15,1,98.60,17.90
3,1,2024-03-16,1,263.40,35.43
4,1,2024-03-22,1,70.00,0.37
5,1,2024-03-23,1,54.80,13.62
6,1,2024-03-26,1,80.00,11.80
7,2,2024-03-09,1,51.30,14.31
8,2,2024-03-20,1,19.10,4.33
9,3,2024-03-01,1,50.50,17.50


In [11]:
run_sql("""
SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10
""")

,pickup_zone_id,n_days
0,1,7
1,2,2
2,3,15
3,4,31
4,6,6
5,7,31
6,8,1
7,9,6
8,10,31
9,11,14


## Завдання 2. Time-series ML features

Усі ознаки рахую віконними функціями на таблиці `t7_hw_taxi_daily`. У кожному вікні є `PARTITION BY pickup_zone_id`, тобто ознаки рахуються окремо для кожної зони. Без цього `LAG` на першому рядку нової зони взяв би значення з останнього рядка попередньої зони, і модель отримала б історію чужої зони.

### Feature 1. Lag-ознаки

`trips_lag_1d`, `trips_lag_7d` і `trips_lag_30d` це кількість поїздок 1, 7 і 30 рядків тому в тій самій зоні.

In [12]:
run_sql("""
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 40
""")

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-03-13,1,NaN,NaN,None
1,1,2024-03-14,1,1.0,NaN,None
2,1,2024-03-15,1,1.0,NaN,None
3,1,2024-03-16,1,1.0,NaN,None
4,1,2024-03-22,1,1.0,NaN,None
5,1,2024-03-23,1,1.0,NaN,None
6,1,2024-03-26,1,1.0,NaN,None
7,2,2024-03-09,1,NaN,NaN,None
8,2,2024-03-20,1,1.0,NaN,None
9,3,2024-03-01,1,NaN,NaN,None


У перших рядках кожної зони lag-ознаки дорівнюють `NULL`, бо попередньої історії ще немає. Для `trips_lag_30d` так буде в перших 30 рядках. У навчальну вибірку такі рядки не беруть, їх відсікають через `WHERE trips_lag_30d IS NOT NULL`.

Є ще одна річ. `LAG` бере попередній **рядок**, а не попередній календарний день. Якщо в якійсь зоні був день без поїздок і рядка за нього немає, то `trips_lag_1d` візьме значення за два дні тому. Щоб такого не було, треба спершу додати в таблицю всі дні, навіть порожні. Я цього не роблю, але пам'ятаю про це обмеження.

### Feature 2. Rolling 7-day average fare

Середня ціна за 7 календарних днів разом із поточним. Для цього `RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW`. Колонку `ts_day` приводжу до `TIMESTAMP`, бо для `RANGE` з `INTERVAL` PostgreSQL потрібен саме такий тип. Колонка `days_in_window` показує, скільки днів реально потрапило у вікно.

In [13]:
run_sql("""
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
0,1,2024-03-13,117.00,117.0000000000000000,1
1,1,2024-03-14,29.60,73.3000000000000000,2
2,1,2024-03-15,98.60,81.7333333333333333,3
3,1,2024-03-16,263.40,127.1500000000000000,4
4,1,2024-03-22,70.00,166.7000000000000000,2
5,1,2024-03-23,54.80,62.4000000000000000,2
6,1,2024-03-26,80.00,68.2666666666666667,3
7,2,2024-03-09,51.30,51.3000000000000000,1
8,2,2024-03-20,19.10,19.1000000000000000,1
9,3,2024-03-01,50.50,50.5000000000000000,1


### Feature 3. Rolling 30-day volatility

Стандартне відхилення середньої ціни за 30 днів (`STDDEV_SAMP`). У першому рядку зони воно `NULL`, бо одне значення не дає відхилення.

In [14]:
run_sql("""
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
0,1,2024-03-13,117.00,None,1
1,1,2024-03-14,29.60,61.8011326757042536,2
2,1,2024-03-15,98.60,46.0766028840379392,3
3,1,2024-03-16,263.40,98.3161397397870445,4
4,1,2024-03-22,70.00,88.8975365237980034,5
5,1,2024-03-23,54.80,83.3112157315368113,6
6,1,2024-03-26,80.00,76.6638425980902806,7
7,2,2024-03-09,51.30,None,1
8,2,2024-03-20,19.10,22.7688383542068303,2
9,3,2024-03-01,50.50,None,1


### Feature 4. Expanding cumulative trips

Нагромаджена сума поїздок і середнє від початку історії зони. Тут вікно `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`, бо потрібні всі рядки від початку до поточного. `ROWS` рахує рядки, а `RANGE` рахує за значенням у `ORDER BY`. Тому для rolling-вікон за днями я брала `RANGE` з інтервалом, а тут `ROWS`.

In [15]:
run_sql("""
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-03-13,1,1,1.00000000000000000000
1,1,2024-03-14,1,2,1.00000000000000000000
2,1,2024-03-15,1,3,1.00000000000000000000
3,1,2024-03-16,1,4,1.00000000000000000000
4,1,2024-03-22,1,5,1.00000000000000000000
5,1,2024-03-23,1,6,1.00000000000000000000
6,1,2024-03-26,1,7,1.00000000000000000000
7,2,2024-03-09,1,1,1.00000000000000000000
8,2,2024-03-20,1,2,1.00000000000000000000
9,3,2024-03-01,1,1,1.00000000000000000000


### Усі ознаки в одному view

Збираю всі чотири групи ознак у `v_t7_hw_taxi_features`. Кожне вікно оголошене з `PARTITION BY pickup_zone_id`. Рядки з `NULL` у `trips_lag_30d` відкидаю, тому що вони не мають повної історії для навчання. Вікна рахуються до цього фільтра, тож rolling і expanding бачать усю історію зони.

In [16]:
run_sql("""
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        AVG(avg_fare)         OVER w7  AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;
""")

In [17]:
run_sql("""
SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,4,2024-03-31,57,19.55,188.71,61,58,26,17.6857142857142857,3.1479601313399360,986,31.8064516129032258
1,7,2024-03-31,17,25.36,91.77,13,16,11,25.0328571428571429,7.1856903219035852,302,9.7419354838709677
2,10,2024-03-31,3,31.28,20.63,5,7,7,45.6028571428571429,10.1732105037703308,190,6.1290322580645161
3,13,2024-03-31,49,22.69,223.55,65,53,95,23.4714285714285714,1.7060770690438464,2925,94.3548387096774194
4,24,2024-03-31,35,14.43,85.93,42,32,52,16.4457142857142857,1.5994775008927350,1281,41.3225806451612903
5,25,2024-03-31,12,25.98,59.34,6,9,11,25.4371428571428571,7.4413812214372286,249,8.0322580645161290
6,33,2024-03-31,10,32.00,71.52,9,11,6,24.8528571428571429,4.3321081910901469,313,10.0967741935483871
7,39,2024-03-31,3,18.10,7.65,5,2,5,32.4642857142857143,9.1984822336241347,131,4.2258064516129032
8,41,2024-03-31,62,16.42,185.93,46,39,52,17.1185714285714286,1.9703358719456594,1406,45.3548387096774194
9,42,2024-03-31,39,16.26,129.41,26,20,19,20.5985714285714286,3.3168829710555302,721,23.2580645161290323


## Завдання 3. JSONB-таблиця для подій

Створюю синтетичну таблицю `t7_hw_events`, це event-log для ML. У ній 5 типів подій: `page_view`, `click`, `add_to_cart`, `checkout` і `recommendation_shown`. Для кожного типу `payload` має свою структуру, наприклад у `checkout` є сума і спосіб оплати, а в `recommendation_shown` масив рекомендацій. Генерую 50 000 рядків.

In [18]:
run_sql("""
DROP TABLE IF EXISTS t7_hw_checkouts_norm;
DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER     NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT        NOT NULL,
    payload     JSONB       NOT NULL
);
""")

In [19]:
run_sql("""
INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    MOD(n, 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00' - MOD(n, 60) * INTERVAL '1 day' AS event_ts,
    CASE MOD(n, 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE 'recommendation_shown'
    END AS event_type,
    CASE MOD(n, 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || (MOD(n, 200) + 1),
            'referrer', (ARRAY['google', 'direct', 'email', 'fb'])[MOD(n, 4) + 1],
            'device', (ARRAY['mobile', 'desktop', 'tablet'])[MOD(n, 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || (MOD(n, 50) + 1),
            'page_url', '/category/' || (MOD(n, 20) + 1),
            'device', (ARRAY['mobile', 'desktop'])[MOD(n, 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', MOD(n, 200) + 1,
            'price', (MOD(n, 100) + 10)::NUMERIC(10, 2),
            'quantity', MOD(n, 5) + 1,
            'tags', jsonb_build_array((ARRAY['promo', 'new', 'clearance'])[MOD(n, 3) + 1])
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', (MOD(n, 500) + 50)::NUMERIC(10, 2),
            'items_count', MOD(n, 10) + 1,
            'payment_method', (ARRAY['card', 'paypal', 'crypto'])[MOD(n, 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || (MOD(n, 5) + 1),
            'model_version', '1.' || MOD(n, 10),
            'recommendations', jsonb_build_array(
                MOD(n, 200) + 1,
                MOD(n + 1, 200) + 1,
                MOD(n + 2, 200) + 1
            ),
            'experiment', 'exp_' || (MOD(n, 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;
""")

In [20]:
run_sql("""
SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type
""")

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


In [21]:
run_sql("""
SELECT event_type, payload
FROM (
    SELECT DISTINCT ON (event_type) event_type, payload
    FROM t7_hw_events
    ORDER BY event_type, event_id
) AS first_events
""")

,event_type,payload
0,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,checkout,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
2,click,"{'device': 'desktop', 'page_url': '/category/2..."
3,page_view,"{'url': '/product/6', 'device': 'tablet', 'ref..."
4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


### JSONB Query 1. Containment `@>`

Шукаю checkout-події, де спосіб оплати `card`.

In [22]:
run_sql("""
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10
""")

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
1,18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items..."
2,33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items..."
3,48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items..."
4,63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'item..."
5,78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'item..."
6,93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'item..."
7,108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'item..."
8,123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'item..."
9,138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'item..."


### JSONB Query 2. Існування ключа `?` та `?|`

`?` перевіряє, чи є в `payload` такий ключ верхнього рівня. `?|` перевіряє, чи є хоча б один ключ зі списку.

In [23]:
run_sql("""
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10
""")

,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3..."
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity':..."
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quanti..."
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3..."
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity':..."
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quanti..."
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3..."
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity':..."
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quanti..."


In [24]:
run_sql("""
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10
""")

,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


### JSONB Query 3. JSONPath

`jsonb_path_query` розгортає масив рекомендацій у окремі рядки. `jsonb_path_exists` перевіряє умову всередині JSON, тут шукаю чекаути з сумою понад 300.

In [25]:
run_sql("""
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15
""")

,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


In [26]:
run_sql("""
SELECT
    event_id,
    payload ->> 'order_id'              AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10
""")

,event_id,order_id,total
0,253,o_100253,303.00
1,258,o_100258,308.00
2,263,o_100263,313.00
3,268,o_100268,318.00
4,273,o_100273,323.00
5,278,o_100278,328.00
6,283,o_100283,333.00
7,288,o_100288,338.00
8,293,o_100293,343.00
9,298,o_100298,348.00


## Завдання 4. GIN-індекс і EXPLAIN ANALYZE

Дивлюсь, як індекси впливають на запити по JSONB. Усі `EXPLAIN (ANALYZE, BUFFERS)` друкую повністю, щоб плани лишились у виводі.

### Step 1. Containment без GIN-індексу

Спочатку видаляю індекси, якщо вони лишились від попереднього запуску, і оновлюю статистику.

In [27]:
run_sql("""
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;

ANALYZE t7_hw_events;
""")

In [28]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card')
""")

Aggregate  (cost=1758.23..1758.24 rows=1 width=8) (actual time=106.217..106.219 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=3293 width=0) (actual time=0.591..105.716 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=7
Planning Time: 0.249 ms
Execution Time: 106.251 ms


### Step 2. GIN-індекс для containment і JSONPath

Створюю GIN-індекс з `jsonb_path_ops`. Він менший і швидший за стандартний `jsonb_ops`, але підтримує тільки `@>` і JSONPath-оператори, а запити з `?` і `?|` не прискорить. Якби мені були потрібні `?` і `?|`, я взяла б стандартний `jsonb_ops`.

In [29]:
run_sql("""
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);

ANALYZE t7_hw_events;
""")

In [30]:
run_sql("""
SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops')) AS gin_index_size,
       pg_size_pretty(pg_relation_size('t7_hw_events'))                             AS table_size
""")

,gin_index_size,table_size
0,1144 kB,8000 kB


### Step 3. Той самий containment після GIN

Повторюю перший запит і додаю ще один, по `referrer`. Цей ключ є лише у `page_view`.

In [31]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card')
""")

Aggregate  (cost=1072.70..1072.71 rows=1 width=8) (actual time=15.205..15.207 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=26.64..1066.12 rows=2632 width=0) (actual time=1.176..14.823 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..25.99 rows=2632 width=0) (actual time=1.000..1.001 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=16
Planning Time: 0.315 ms
Execution Time: 15.283 ms


In [32]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google')
""")

Aggregate  (cost=1061.31..1061.32 rows=1 width=8) (actual time=10.746..10.748 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=24.01..1055.98 rows=2131 width=0) (actual time=2.529..10.480 rows=2500 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..23.48 rows=2131 width=0) (actual time=2.322..2.322 rows=2500 loops=1)
              Index Cond: (payload @> jsonb_build_object('referrer', 'google'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.198 ms
Execution Time: 10.788 ms


### Step 4. Коли GIN не допомагає

GIN на всю колонку `payload` не допоможе для запиту `payload ->> 'device' = 'mobile'`, бо тут порівнюється текст з одного поля, а не вкладений JSON. Для такого запиту потрібен expression B-tree індекс. Спочатку запускаю запит без нього (GIN уже є), потім створюю індекс і повторюю.

In [33]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile'
""")

Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=59.756..59.758 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.020..58.511 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.076 ms
Execution Time: 59.788 ms


In [34]:
run_sql("""
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));

ANALYZE t7_hw_events;
""")

In [35]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile'
""")

Aggregate  (cost=1240.86..1240.87 rows=1 width=8) (actual time=12.814..12.816 rows=1 loops=1)
  Buffers: shared hit=1000 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=96.30..1220.21 rows=8260 width=0) (actual time=0.721..12.091 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1000 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..94.24 rows=8260 width=0) (actual time=0.534..0.535 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=19 read=1
Planning Time: 0.301 ms
Execution Time: 12.867 ms


## Завдання 5. JSONB проти нормалізованої таблиці

Для прикладу беру подію `checkout`.

### Step 1. Нормалізована таблиця для checkout-подій

Роблю таблицю `t7_hw_checkouts_norm` з окремими колонками і обмеженнями (`CHECK`) та переношу туди checkout-події з JSONB.

In [36]:
run_sql("""
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER     NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT        NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER     NOT NULL CHECK (items_count > 0),
    payment_method  TEXT        NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id, user_id, event_ts, order_id, total, items_count, payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id',
    (payload ->> 'total')::NUMERIC(10, 2),
    (payload ->> 'items_count')::INTEGER,
    payload ->> 'payment_method'
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;
""")

In [37]:
run_sql("""
SELECT COUNT(*) AS n_checkouts
FROM t7_hw_checkouts_norm
""")

,n_checkouts
0,10000


### Step 2. Один запит двома способами

Хочу порахувати середню суму чекаута за способом оплати і взяти топ-3. Спочатку JSONB-версія.

In [38]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method'          AS payment_method,
    AVG((payload ->> 'total')::NUMERIC)   AS avg_total,
    COUNT(*)                              AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3
""")

Limit  (cost=1957.21..1957.21 rows=3 width=72) (actual time=26.598..26.600 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1957.21..1971.17 rows=5587 width=72) (actual time=26.596..26.597 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1801.19..1885.00 rows=5587 width=72) (actual time=26.546..26.582 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1650.17 rows=10068 width=132) (actual time=0.032..16.055 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning:
  Buffers: shared hit=15 read=1
Planning Time: 0.223 ms
Execution Time: 32.66

Тепер те саме по нормалізованій таблиці.

In [39]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*)   AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3
""")

Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=32.145..32.148 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=32.143..32.145 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=32.120..32.124 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.012..1.531 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning:
  Buffers: shared hit=3
Planning Time: 0.157 ms
Execution Time: 32.192 ms


### Step 3. Бонус: MongoDB-версія запиту

Це не код для PostgreSQL, а псевдокод для порівняння.

```javascript
// MongoDB: середня сума чекаута за способом оплати, топ-3
// Псевдокод, у PostgreSQL не виконується

db.events.aggregate([
  {$match: {event_type: "checkout"}},
  {$group: {
    _id: "$payload.payment_method",
    avg_total: {$avg: {$toDouble: "$payload.total"}},
    n: {$sum: 1}
  }},
  {$sort: {avg_total: -1}},
  {$limit: 3}
]);
```

Різниця між підходами, як я її розумію:
- PostgreSQL з JSONB зручний, коли потрібні SQL, `JOIN`, транзакції і зв'язок із нормалізованими таблицями;
- aggregation pipeline в MongoDB природний, коли дані зберігаються як документи;
- великий потік сирих кліків краще може жити в document store або data lake;
- готові ML-ознаки зазвичай зручніше тримати в звичайних SQL-таблицях з типами.

## Завдання 6. Reflection

### 1. Що змінилось у плані після GIN

Без індексу запит по `payment_method` ішов через `Seq Scan`: PostgreSQL прочитав усі 1000 сторінок таблиці, cost був близько 1758, виконання близько 106 мс. Після GIN з'явились `Bitmap Index Scan` і `Bitmap Heap Scan`, cost впав приблизно до 1073, а час до 15 мс. Запит по `referrer` теж пішов через `Bitmap Index Scan`.

### 2. Чому GIN не допомагає для `payload ->> 'device' = 'mobile'`

GIN відповідає на питання «чи є в JSON такий фрагмент». А `->>` витягує текст одного поля, і для такого порівняння цей індекс не підходить, тому план лишився `Seq Scan` (cost 1751, близько 60 мс). Після expression B-tree індексу з'явився `Bitmap Index Scan`, cost близько 1241, час близько 13 мс. Оцінка рядків також стала точнішою: було 250, стало 8260 при реальних 8333.

### 3. Trade-off GIN

Індекс займає 1144 kB при розмірі таблиці 8000 kB, тобто близько 14%. Читання швидшає, але кожен `INSERT` чи `UPDATE` має оновлювати індекс, тому запис повільніший. Швидкість запису окремо я не міряла.

### 4. JSONB чи нормалізована таблиця

Для checkout-подій нормалізована таблиця виграла за cost і кількістю сторінок: 269 проти 1957 і 94 проти 1000. За загальним часом різниці майже немає, 32,2 мс проти 32,7 мс. Сам `Seq Scan` по нормалізованій таблиці швидший (1,5 мс проти 16 мс), але на такому малому обсязі час сильно коливається, тому я більше довіряю cost і сторінкам. У JSONB PostgreSQL читає всі події і з кожного рядка дістає поля. JSONB я б брала для сирого event-log, де структура подій різна і змінюється. Нормалізовану таблицю візьму для ознак, які модель використовує постійно, бо там є типи й обмеження.

### 5. Новий тип `refund`

У JSONB достатньо почати писати події з `event_type = 'refund'` і своїм `payload`, схему міняти не треба. У нормалізованому варіанті треба створити нову таблицю з потрібними колонками, перенести вже наявні дані, якщо вони є, і змінити код, який з нею працює. Якщо повернення мають бути пов'язані із замовленням, у таблиці знадобиться `order_id`.